# Synthetic routing: hydraulic applicability audit

Use the completed grid to examine hydraulic assumptions before transferring its conclusions to real rivers. This notebook reads saved jobs and results; it does **not** rerun or modify routing. Static figures remain visible in notebook previews.

The wave-regime framework is from [Moussa & Bocquillon (1996)](https://doi.org/10.1016/S0022-1694(96)03045-4); see the original diagram reproduced in **Figure 1** of [Getirana & Paiva (2013), PDF](https://gmao.gsfc.nasa.gov/media/gmaoftp/sarith/ROUTING_MODEL/docs/amazon_wrcr20212.pdf#page=2). Their coordinates are

$$F_0^2=U_0^2/(g h_0),\qquad T^+=T_0 U_0 S_{f0}/h_0.$$

The literature uses a steady reference flow and perturbation period. Here rise duration is an explicit **proxy**, with duration sensitivity below. The prescribed reach slope substitutes for friction slope, which routing does not solve. Our plots reproduce these coordinates and overlay the **repository's provisional screen** ($T^+\ge80$, $F_0^2\le0.9$), not the complete published wave-zone boundaries. Failing this screen does not identify a particular alternative wave model. Passing it does not validate RAPID or demonstrate absence of backwater.

Depth is inferred as $h=W/k_b$, area as $Wh$, and velocity as $Q/A$. This is a rectangular reference geometry, not a simulated water depth. Routing celerity is kept separate from water velocity. The fixed-depth peak scenario is sensitivity only, not a steady reference or a routed internal peak.

This version includes **nested/main_island_only**: collapse the main island while retaining the upper inner branch and its length adjustment. Results come from `full_grid_island`; the original six-comparison grid remains in `full_grid`.

**Checkpoint · 29 September 2026:** [SYN-2026-09-29](../notes/2026-09-29_synthetic_checkpoint.md) records the findings, the pause pending real-case comparison, and the restart of hydraulic applicability.

**Current decision · 30 September 2026:** synthetic depth scaling uses **α=0.65**, anchored to the BASED reference. The focused results below include this selection. **Real-case selection waits for new runs using updated slopes.** The full grid and extension have now been rerun in a separate selected version; the final section reports their current diagnostics. Selection: [selected_hydraulic_reference.json](../configs/selected_hydraulic_reference.json).


**Full selected rerun now available:** see the final section and the current synthesis/predictor notebooks. Earlier hydraulic audits in this notebook retain their explicitly dated settings as diagnostic history; they are not silently converted to the selected parameterization.


**5 October 2026 update:** the current real-network audit is immediately below. Later statements about awaiting real runs belong to the dated historical investigations; the 16 production base networks are now complete. The current audit does not establish improved applicability from faster routing alone.

## Current real-network placement · 5 October 2026

Reference **CURRENT-HYDRAULIC-DOMAIN-2026-10-05**. This section replaces the historical gauge-event overlay as the evidence for the actual 16 study networks. It uses 5,315 physical base links from completed production inputs: final slopes, BASED-anchored alpha=.65 depths, n=.035, and recorded raw/used celerities. The subsequent historical sections retain their dated settings.

Water velocity is either steady width-allocated Q/(W h), or raw Manning celerity times 3/5. **Capped routing celerity is not substituted for water velocity.** Peak-Q coordinates keep reference depth fixed; these are sensitivity scenarios, not routed internal peaks. Very high allocated-flow Froude values indicate reference-state inconsistency, not simulated high-speed flow.

The wave-regime coordinates depend on velocity, depth, slope and a perturbation period, as in [Getirana & Paiva (2013), equations 3–4](https://gmao.gsfc.nasa.gov/media/gmaoftp/sarith/ROUTING_MODEL/docs/amazon_wrcr20212.pdf). A faster routing result does not itself determine this placement. The green region remains the repository's provisional screen, not the complete literature boundary or proof of RAPID validity.

**Duration uncertainty is material.** We show first forcing 1% departure-to-first-peak (a broad event-window proxy) and the stored local-minimum-to-peak (a potentially very short shoulder proxy). Neither is adopted as the true perturbation period. Broad departures can span multiple rises, especially example 48. These alternatives do not change the accepted parent–child response metrics. Synthetic points use their prescribed 12 h rise; this differs slightly from a 1% departure for a triangular rise.

The broad-window figure below gives real baseflow passing-length fractions of **16.3% allocated / 17.6% Manning**, versus approximately **0.05% / 0.04%** with the stored local rise. Those are pooled physical-length fractions, not percentages of valid rivers. They demonstrate sensitivity, not a basis for choosing whichever duration passes more links. No matched old/new hydraulic reconstruction was made, so this does not establish improvement over the old parameterization.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
DOMAIN_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
DOMAIN_OUT = DOMAIN_ROOT/'RAPID/validation/current_hydraulic_domain_20261005'
domain_summary = pd.read_csv(DOMAIN_OUT/'summary.csv', dtype={'example': str})
domain_provenance = json.loads((DOMAIN_OUT/'provenance.json').read_text())
display(pd.DataFrame(domain_provenance['sources'])[['example','rise_hours','legacy_local_rise_hours']].round(2))
display(domain_summary[domain_summary.example.eq('all')].round(5))
display(Image(filename=str(DOMAIN_OUT/'current_domain.png')))
# Full per-network results remain available without forcing 128 rows into the main view.
display(domain_summary.query("example != 'all' and reference_flow == 'base' and velocity == 'manning'").pivot(index='example', columns='time_proxy', values='passing_length_fraction').round(4))


**Interpretation:** updated synthetic and real coordinates overlap partially, while real networks occupy a much wider range. The previous gauge-screening cohort is not a matched historical baseline. Current results do not yet support “the faster model is now in the correct hydraulic domain.” Resolve a defensible event-period proxy and the Q/depth/Manning consistency diagnostics before making that claim. Keep agreement with observed hydrographs as a separate validation question.

Reproduce with `RAPID/investigations/current_hydraulic_domain.py`. Source identities and formulas are in [the audit provenance](../../RAPID/validation/current_hydraulic_domain_20261005/provenance.json).


### Roughness-only comparison · 5 October 2026

The legacy original synthetic manifest confirms n=.35; current production uses .035. At fixed depth/slope, raw Manning velocity scales as 1/n, so T⁺ scales as 1/n and F² as 1/n². Applying n=.35 counterfactually to the **current real base networks**, with the same broad-rise durations, gives **2.79%** passing channel length; current n=.035 gives **17.57%**. Thus the roughness change alone moves more current real links into the provisional screen. This is not a matched historical rerun: old slopes/depths, event definitions and cohorts also differed. Clipped routing celerity is not used as water velocity. Allocated-Q coordinates do not depend directly on n at fixed Q/width/depth/slope.

For the current **original synthetic grid**, Manning-reference passing length is **0%**, and allocated baseflow passing length is **0%**. The allocated peak-Q scenario reaches **0.643%**, at fixed reference depth. The current figure excludes the wider embedding extension. These results limit physical interpretation of the synthetic routing experiments; passing/failing this provisional screen is not a full wave-model classification. Do not tune roughness or event period merely to place points inside it. Source: `roughness_only_counterfactual.csv` and the selected synthetic hydraulic summary.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'synthetic_experiments').is_dir())
EXPERIMENT = 'full_grid_island'
RUN = ROOT / 'synthetic_experiments' / 'outputs' / EXPERIMENT
OUT = RUN / 'applicability'
OUT.mkdir(parents=True, exist_ok=True)
REAL_PATH = ROOT / 'SWORD_gauge_match' / 'outputs' / 'kinematic_results.parquet'
REAL_KB = 20.0
REAL_Q0_METHOD = 'pre_event_median'
REAL_T0_METHOD = 'rise_start_to_peak'
# Editable diagnostic thresholds; these do not filter the simulation results.
TPLUS_MIN, F2_MAX = 80.0, 0.9
G = 9.80665
pd.set_option('display.max_columns', 20)
manifest = json.loads((RUN / 'manifest.json').read_text())
states = pd.read_csv(RUN / 'states.csv')
results = pd.read_csv(RUN / 'results.csv')
print(f"{len(states):,} state references; {states.run_id.nunique():,} unique runs; {len(results):,} comparisons")
print('Saved plan:', manifest['plan_id'], '| saved source:', manifest['source'])
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})


## Reach reference hydraulics

Propagate inlet discharge through each saved directed network using its width allocation rule. Baseflow allocation is a steady diagnostic; the peak allocation does not include branch-dependent attenuation or travel time. Parallel edges remain separate. Each row represents a canonical physical reach before segmentation, avoiding overcounting long reaches merely because they contain more 200 m segments.

The Manning velocity below uses the same **wide-channel approximation** as the routing celerity formula: $U_M=h^{2/3}\sqrt{S}/n$, $c_{raw}=5U_M/3$. Comparing $Q/A$ with $U_M$ exposes incompatible reference states. Celerity clipping is shown separately.

In [ ]:
rows = []
for run_id in sorted(states.run_id.unique()):
    job = json.loads((RUN / 'jobs' / f'{run_id}.json').read_text())
    net, settings = job['network'], job['settings']
    edges = net['edges']
    graph = nx.DiGraph()
    graph.add_nodes_from(range(len(net['nodes'])))
    graph.add_edges_from((e[0], e[1]) for e in edges)
    sources = [v for v in graph if graph.in_degree(v) == 0]
    assert len(sources) == 1 and nx.is_directed_acyclic_graph(graph)
    node_flow = dict.fromkeys(graph, 0.0)
    node_flow[sources[0]] = 1.0
    shares = {}
    for u in nx.topological_sort(graph):
        outgoing = [(i, e) for i, e in enumerate(edges) if e[0] == u]
        total_width = sum(e[3] for _, e in outgoing)
        for i, e in outgoing:
            share = node_flow[u] * e[3] / total_width
            shares[i] = share
            node_flow[e[1]] += share
    assert np.isclose(sum(node_flow[v] for v in graph if graph.out_degree(v) == 0), 1)
    for i, (u, v, length, width, slope) in enumerate(edges):
        depth = width / settings['kb']
        area = width * depth
        um = depth**(2/3) * np.sqrt(slope) / settings['n_manning']
        c_raw = 5/3 * um
        c_used = np.clip(c_raw, settings['min_celerity_mps'], settings['max_celerity_mps'])
        for scenario, inlet_q in [('baseflow', settings['baseflow_cms']), ('peak reference', settings['peak_cms'])]:
            q = shares[i] * inlet_q
            velocity = q / area
            rows.append(dict(run_id=run_id, edge=i, u=u, v=v, length_m=length,
                width_m=width, slope=slope, depth_m=depth, flow_share=shares[i],
                q_cms=q, velocity_mps=velocity, manning_velocity_mps=um,
                q_over_manning_q=velocity/um, celerity_raw_mps=c_raw, celerity_used_mps=c_used,
                capped=not np.isclose(c_raw, c_used), scenario=scenario,
                rise_hours=settings['rise_hours'], fall_hours=settings['fall_hours'],
                f2=velocity**2/(G*depth), tplus=settings['rise_hours']*3600*velocity*slope/depth))
reach = pd.DataFrame(rows)
reach['screen_pass'] = reach.tplus.ge(TPLUS_MIN) & reach.f2.le(F2_MAX)
assert np.isfinite(reach[['f2', 'tplus', 'q_over_manning_q']]).all().all()
base = reach.loc[reach.scenario.eq('baseflow')].copy()
# Analytic sanity check for the baseline centerline: W=500 m, kb=20, Q=100, S=1e-4.
assert np.isclose((100/(500*25))**2/(G*25), 2.61047478458e-7)
assert np.isclose(12*3600*(100/(500*25))*1e-4/25, 0.0013824)
display(reach.groupby('scenario').agg(reach_rows=('edge','size'),
    min_Tplus=('tplus','min'), median_Tplus=('tplus','median'), max_Tplus=('tplus','max'),
    screen_pass_fraction=('screen_pass','mean'), capped_fraction=('capped','mean')))


## Real-river comparison and provenance

Read the gauge/SWORD screening product with the same $k_b$, pre-event discharge, and start-to-peak rise proxy. Keep selected events and best matched reaches. This is a **gauge screening cohort**, not a verified list of the final RAPID case-study networks. Repeated events are retained; distributions are event-weighted. No regime-based filtering is used.

`froude_f0` in the existing product is **squared** Froude; verify it directly against stored velocity and depth. Missing files produce an explicit empty overlay rather than fabricated observations.

In [ ]:
real = pd.DataFrame(columns=['f2', 'tplus'])
real_audit = {'source': str(REAL_PATH), 'cohort': 'selected events, best gauge/reach matches'}
required_real = [REAL_PATH, REAL_PATH.parent/'events_selected.parquet', REAL_PATH.parent/'crosswalk_best.parquet']
if all(p.exists() for p in required_real):
    cols = ['station_key','event_id','reach_id','kb','q0_method','t0_method',
            'velocity_v0_ms','depth_y0_m','slope_value','t0_hours','froude_f0','tplus','valid_input']
    real = pd.read_parquet(REAL_PATH, columns=cols, filters=[
        ('kb','==',REAL_KB), ('q0_method','==',REAL_Q0_METHOD), ('t0_method','==',REAL_T0_METHOD)])
    real_audit['rows_after_method_filter'] = len(real)
    selected = pd.read_parquet(required_real[1], columns=['station_key','event_id']).drop_duplicates()
    best = pd.read_parquet(required_real[2], columns=['station_key','reach_id']).dropna().drop_duplicates()
    real = real.merge(selected, on=['station_key','event_id'], how='inner', validate='many_to_one')
    real = real.merge(best, on=['station_key','reach_id'], how='inner', validate='many_to_one')
    real_audit['rows_before_validity_filter'] = len(real)
    real = real.rename(columns={'froude_f0':'f2'})
    good = real.valid_input.fillna(False) & np.isfinite(real[['f2','tplus']]).all(axis=1) & real.f2.gt(0) & real.tplus.gt(0)
    real_audit['excluded_invalid_or_nonpositive'] = int((~good).sum())
    real = real.loc[good].copy()
    assert np.allclose(real.f2, real.velocity_v0_ms**2/(G*real.depth_y0_m))
    assert np.allclose(real.tplus, real.t0_hours*3600*real.velocity_v0_ms*real.slope_value/real.depth_y0_m)
    real['screen_pass'] = real.tplus.ge(TPLUS_MIN) & real.f2.le(F2_MAX)
    real_audit.update(rows=len(real), stations=real.station_key.nunique(),
        screen_pass_fraction=float(real.screen_pass.mean()))
else:
    real_audit['status'] = 'Missing real-river inputs; synthetic-only analysis.'
display(pd.Series(real_audit, name='Real-data audit').to_frame())


## Wave-regime coordinates

The green area denotes only the provisional repository screen. Both axes are logarithmic so very low Froude numbers remain visible. Distinct synthetic coordinates are plotted once; overlapping grid configurations do not imply additional hydraulic evidence. Real points are deterministically sampled for rendering only; all valid rows enter summaries.

In [ ]:
def regime_axes(ax):
    ax.set(xscale='log', yscale='log', xlabel=r'$T^+$ (rise-duration proxy)', ylabel=r'$F_0^2$')
    ax.axvline(TPLUS_MIN, color='#15803d', ls='--', lw=1)
    ax.axhline(F2_MAX, color='#15803d', ls=':', lw=1)
    ax.grid(alpha=.15)

def shade_screen(ax):
    xmin, xmax = ax.get_xlim(); ymin, ymax = ax.get_ylim()
    xmax = max(xmax, TPLUS_MIN*10); ymax = max(ymax, F2_MAX*2)
    ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
    ax.fill_between([TPLUS_MIN,xmax], ymin, F2_MAX, color='#dcfce7', zorder=0)

fig, axes = plt.subplots(1,2,figsize=(13,5), constrained_layout=True)
real_plot = real.sample(min(len(real),40000), random_state=0) if len(real) else real
for ax, scenario in zip(axes, ['baseflow','peak reference']):
    regime_axes(ax)
    if len(real_plot):
        ax.scatter(real_plot.tplus, real_plot.f2, s=5, color='#94a3b8', alpha=.18, rasterized=True, label='Real gauge-event cohort (baseflow)')
    points = reach.loc[reach.scenario.eq(scenario), ['tplus','f2']].drop_duplicates()
    ax.scatter(points.tplus, points.f2, s=16, color='#2563eb', alpha=.5, label=f'Synthetic: {scenario}')
    shade_screen(ax)
    ax.set_title(scenario.capitalize(), loc='left'); ax.legend(fontsize=8, loc='lower right')
fig.suptitle('Hydraulic coordinates · shaded region = provisional repository screen')
fig.savefig(OUT/'regime_overlay.png', dpi=180)
display(fig); plt.close(fig)


## Baseline and hydraulic consistency

A match between $Q/A$ and Manning velocity would support a common reference state under the assumed geometry, roughness and slope. A mismatch is an assumption diagnostic, not a solver failure. The $3/5$ conversion applies to the **raw Manning-derived** celerity; clipping that celerity is numerical regularization and does not define a new physical water velocity.

In [ ]:
configs = json.loads((RUN/'configurations.json').read_text())
baseline_parameters = dict(total_width_m=500, reference_slope=1e-4, outer_a_fraction=.8,
    inner_fraction=.5, length_multiplier=1., total_length_m=20000, internal_location='central')
ids = [c['configuration_id'] for c in configs if c['parameters'] == baseline_parameters]
if ids:
    baseline_states = states.loc[states.configuration_id.eq(ids[0])]
    baseline_table = baseline_states.merge(reach, on='run_id')
    display(baseline_table.loc[baseline_table.state_id.eq('centerline/original'),
        ['scenario','width_m','depth_m','q_cms','velocity_mps','manning_velocity_mps',
         'q_over_manning_q','celerity_raw_mps','celerity_used_mps','f2','tplus','screen_pass']])
else:
    print('Baseline configuration not present in this experiment.')
fig, axes = plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)
for scenario, group in reach.groupby('scenario'):
    unique = group[['velocity_mps','manning_velocity_mps','q_over_manning_q']].drop_duplicates()
    axes[0].scatter(unique.manning_velocity_mps, unique.velocity_mps, s=12, alpha=.3, label=scenario)
    axes[1].hist(np.log10(group.q_over_manning_q), bins=60, histtype='step', density=True, label=scenario)
limits = [min(reach.velocity_mps.min(),reach.manning_velocity_mps.min()), max(reach.velocity_mps.max(),reach.manning_velocity_mps.max())]
axes[0].plot(limits,limits,'k--',lw=1,label='Consistent reference state')
axes[0].set(xscale='log',yscale='log',xlabel='Manning reference velocity (m/s)',ylabel='Allocated Q / area (m/s)')
axes[1].axvline(0,color='k',ls='--',lw=1)
axes[1].set(xlabel=r'$\log_{10}[(Q/A)/U_M]$',ylabel='Density across unique-run reach rows')
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.15)
fig.savefig(OUT/'hydraulic_consistency.png',dpi=180)
display(fig); plt.close(fig)


## Time-scale sensitivity

For the triangular forcing, the 10–90% rising interval is 0.8 times the rise duration. The full pulse duration is rise plus fall. Neither is a unique equivalent sinusoidal period. These alternatives shift $T^+$ while leaving $F_0^2$ unchanged; the required duration shows how far each reference state lies from the provisional $T^+=80$ threshold.

In [ ]:
base['required_T_hours'] = TPLUS_MIN * base.depth_m / (base.velocity_mps * base.slope) / 3600
sensitivity = []
for label, multiplier in [('10–90% rise', np.full(len(base),.8)), ('Full rise', np.ones(len(base))),
                          ('Full pulse', (base.rise_hours+base.fall_hours)/base.rise_hours)]:
    tp = base.tplus * multiplier
    sensitivity.append(dict(time_proxy=label, min_Tplus=tp.min(), median_Tplus=tp.median(), max_Tplus=tp.max(),
        reach_screen_pass_fraction=float((tp.ge(TPLUS_MIN)&base.f2.le(F2_MAX)).mean())))
display(pd.DataFrame(sensitivity))
display(base.required_T_hours.quantile([0,.25,.5,.75,1]).rename('Hours required for Tplus=80').to_frame())

if not base.screen_pass.any():
    display(Markdown("**Finding:** no synthetic baseflow reaches pass the provisional screen under these reference-state assumptions. This diagnoses a gap between the current design and that screen; it does not invalidate the numerical routing or establish another wave regime."))


## Attach diagnostics to parent–child results

For each run, report the channel-length fraction passing the provisional screen. Requiring **all** reaches to pass is an intentionally strict summary, not a validated network-level criterion. The comparison table preserves every result and records both parent and child diagnostics. No observations are discarded for failing the screen. Celerity-cap and time-resolution flags retain the separate numerical concerns.

In [ ]:
base['passing_length_m'] = base.length_m * base.screen_pass
run_audit = base.groupby('run_id').agg(all_reaches_pass=('screen_pass','all'),
    min_tplus=('tplus','min'), max_f2=('f2','max'), total_length_m=('length_m','sum'),
    passing_length_m=('passing_length_m','sum'), any_celerity_cap=('capped','any'),
    min_velocity_ratio=('q_over_manning_q','min'), max_velocity_ratio=('q_over_manning_q','max'))
run_audit['passing_length_fraction'] = run_audit.passing_length_m/run_audit.total_length_m
comparison_audit = results.copy()
for side in ['parent','child']:
    comparison_audit = comparison_audit.merge(run_audit.add_prefix(side+'_'),
        left_on=side+'_run_id', right_index=True, validate='many_to_one')
comparison_audit['both_all_reaches_pass'] = comparison_audit.parent_all_reaches_pass & comparison_audit.child_all_reaches_pass
comparison_audit['within_one_timestep'] = comparison_audit.lag_shift_seconds.abs().le(comparison_audit.timing_resolution_seconds)
assert len(comparison_audit) == len(results)
display(comparison_audit.groupby('child_state').agg(comparisons=('configuration_id','size'),
    both_pass=('both_all_reaches_pass','sum'), later_peaks=('lag_shift_seconds',lambda x: int((x>0).sum())),
    within_one_timestep=('within_one_timestep','sum'), median_lag_shift_pct=('lag_shift_pct','median')))
reach.to_csv(OUT/'reach_diagnostics.csv',index=False)
run_audit.to_csv(OUT/'run_diagnostics.csv')
comparison_audit.to_csv(OUT/'comparison_diagnostics.csv',index=False)
(OUT/'provenance.json').write_text(json.dumps(dict(plan_id=manifest['plan_id'], saved_source=manifest['source'],
    real=real_audit, real_kb=REAL_KB, real_q0_method=REAL_Q0_METHOD, real_t0_method=REAL_T0_METHOD,
    tplus_min=TPLUS_MIN, f2_max=F2_MAX, synthetic_time_proxy='rise_hours',
    slope_assumption='prescribed head gradient substitutes for friction slope',
    references=['https://doi.org/10.1016/S0022-1694(96)03045-4','https://doi.org/10.1002/wrcr.20212']), indent=2))
print('Saved diagnostics and figures to', OUT.relative_to(ROOT))


## Interpretation and next decisions

1. **Use this grid for controlled model behavior.** Conservation and numerical stability support the routing calculations; they do not establish hydraulic applicability.
2. **Resolve the reference state before selecting a physically representative grid.** Width, $k_b$, roughness, discharge and slope were prescribed independently. Decide whether depth should come from the observed width–depth relation or from a discharge-consistent hydraulic closure. Revisit forcing duration using the real event distribution.
3. **Keep the completed grid as exploratory evidence.** Report its assumptions alongside the resistance–timing figures. A failed provisional screen is not grounds to silently remove a result or assign a different wave regime.
4. **Verify transfer to the actual case studies.** The real overlay is the gauge screening cohort; join final RAPID examples and their actual routing parameters before claiming case-study coverage. The full published wave-zone classification has not been implemented here.
5. **Then test the mechanism and numerical sensitivity.** Focus on crossover cases with later peaks, capped runs, and shifts near one timestep; check finer segmentation and branch hydrographs before fitting a general relation.

This notebook audits hydraulic assumptions. It does not yet supply the full numerical-convergence study or a hydraulic validation of RAPID.

## Hydraulic restart · 29 September 2026: resolve the reference state first

Reference: [SYN-2026-09-29](../notes/2026-09-29_synthetic_checkpoint.md). The sensitivity/predictor work is paused pending real-case comparison; this section resumes the applicability audit without changing saved routing.

[Getirana & Paiva (2013), equations 3–4 and Figure 1](https://gmao.gsfc.nasa.gov/media/gmaoftp/sarith/ROUTING_MODEL/docs/amazon_wrcr20212.pdf#page=2) define wave-regime coordinates for an unperturbed flow and a perturbation period. The provisional repository thresholds are not an implementation of all published wave zones or a validated RAPID acceptance boundary.

Our current reference uses $h_W=W/k_b$, $U_Q=Q/(Wh_W)$ but routing uses a different velocity through $c_{raw}=(5/3)U_M$, where $U_M=h_W^{2/3}\sqrt{S}/n$. These coincide only if

$$Q=Q_M=\frac{W}{n}h_W^{5/3}\sqrt{S}.$$

The table and figure compare three **diagnostic closures**, all using the same rise-duration proxy:

1. **Saved width depth + allocated Q:** the existing audit, which can be inconsistent with Manning flow.
2. **Width depth + Manning Q:** retain the depth, slope and roughness; calculate the implied discharge and velocity. The implied discharges need not conserve flow across junctions, so this is not a replacement network forcing.
3. **Allocated Q + normal depth:** retain allocated discharge, width, slope and roughness; solve the wide-channel Manning expression for $h_N=(nQ/[W\sqrt{S}])^{3/5}$ and set $U_N=Q/(Wh_N)$. This restores a local uniform-flow relation; it does not solve junction water levels or backwater and would change routing celerity if adopted.

All comparisons use raw hydraulic velocities, never clipped celerity. Peak-reference rows are steady sensitivities, not simulated internal peak discharges. The original grid is the scope here; the embedding extension has not been added to this audit.


In [ ]:
closure_parts=[]
for scenario,raw in reach.groupby('scenario'):
    q=raw.q_cms.to_numpy();w=raw.width_m.to_numpy();slope=raw.slope.to_numpy()
    h=raw.depth_m.to_numpy();um=raw.manning_velocity_mps.to_numpy()
    # Recover the exact saved reach roughness from the audited Manning relation.
    roughness=h**(2/3)*np.sqrt(slope)/um
    normal_depth=(roughness*q/(w*np.sqrt(slope)))**(3/5)
    assert np.allclose(w*normal_depth**(5/3)*np.sqrt(slope)/roughness,q)
    for label,depth,velocity,discharge in [
        ('Saved width depth + allocated Q',h,q/(w*h),q),
        ('Width depth + Manning Q',h,um,w*h*um),
        ('Allocated Q + normal depth',normal_depth,q/(w*normal_depth),q)]:
        c=raw[['run_id','edge','scenario','width_m','slope','length_m','rise_hours']].copy()
        c['closure']=label;c['reference_depth_m']=depth;c['reference_velocity_mps']=velocity
        c['reference_q_cms']=discharge;c['depth_ratio_to_saved']=depth/h
        c['f2']=velocity**2/(G*depth)
        c['tplus']=c.rise_hours*3600*velocity*slope/depth
        c['screen_pass']=(c.tplus>=TPLUS_MIN)&(c.f2<=F2_MAX)
        c['required_period_hours']=TPLUS_MIN*depth/(velocity*slope)/3600
        closure_parts.append(c)
closure=pd.concat(closure_parts,ignore_index=True)
assert np.isfinite(closure[['f2','tplus','reference_q_cms']]).all().all()
closure_summary=closure.groupby(['scenario','closure']).agg(
    reaches=('edge','size'),median_depth_m=('reference_depth_m','median'),
    median_tplus=('tplus','median'),min_tplus=('tplus','min'),max_tplus=('tplus','max'),
    median_f2=('f2','median'),screen_pass_fraction=('screen_pass','mean'),
    median_required_period_hours=('required_period_hours','median'))
display(closure_summary)
if ids:
    baseline_run=baseline_states.loc[baseline_states.state_id.eq('centerline/original'),'run_id'].iloc[0]
    baseline_closure=closure[closure.run_id.eq(baseline_run)].drop_duplicates(['scenario','closure'])
    display(baseline_closure[['scenario','closure','reference_depth_m','reference_velocity_mps',
        'reference_q_cms','tplus','f2','required_period_hours']].round(6))
closure_summary.to_csv(OUT/'reference_closure_summary.csv')
closure.to_parquet(OUT/'reference_closure_diagnostics.parquet',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
for ax,scenario in zip(axes,['baseflow','peak reference']):
    regime_axes(ax)
    for label,g in closure[closure.scenario.eq(scenario)].groupby('closure',sort=False):
        points=g[['tplus','f2']].drop_duplicates()
        ax.scatter(points.tplus,points.f2,s=12,alpha=.35,label=label,rasterized=True)
    shade_screen(ax);ax.set_title(scenario.capitalize());ax.legend(fontsize=7,loc='lower right')
fig.suptitle('Reference-state sensitivity; no routing reruns or changed model parameters')
fig.savefig(OUT/'reference_closure_coordinates.png',dpi=180);display(fig);plt.close(fig)


### What this resolves, and what it leaves open

**Completed diagnostic, 29 September 2026:** at the baseline centerline (width 500 m, slope 1e−4, Manning n=0.35), width-derived depth is 25 m. The imposed 100 m³/s baseflow gives velocity 0.008 m/s; the Manning relation at that depth implies approximately 3,054 m³/s and 0.244 m/s. Holding discharge instead gives normal depth approximately 3.21 m. All three closures still have zero reaches passing the provisional screen at the 12-hour rise proxy, in both reference scenarios. Correcting local hydraulic consistency alone therefore does not close the gap to that screen. This is not a published wave-zone classification.

A different reference closure moves the wave-regime coordinates without changing any saved hydrograph. Therefore we must agree on a defensible reference state before interpreting a point's position as a physical property of the simulation.

Under the width-depth assumption, $Q_M\propto W^{8/3}\sqrt{S}$ at fixed $k_b,n$, whereas the routing bifurcation rule allocates discharge in proportion to outgoing width. For parallel branches of equal slope this makes the discrepancy width-dependent. Simply adjusting the inlet discharge cannot generally make every unequal branch simultaneously consistent with its prescribed width-derived depth. With a common discharge per unit width and slope, the alternative normal depth is constant across those branches instead. Neither local closure establishes network water-level compatibility.

**Next sequence:**

1. Verify the reference-state conventions and parameters actually used in the final real case studies, including whether width-derived depth represents the event's flow depth. The gauge cohort is not a substitute for this check.
2. Resolve the applicable published wave-zone boundaries and period convention; keep the current provisional screen visibly separate. A triangular rise time is not uniquely a perturbation period.
3. Compare representative actual-case and synthetic reference states under explicitly matched conventions. Inspect baseflow and representative event flow separately.
4. Only then specify a small, hydraulically coherent routing sensitivity experiment if needed. Do not retune the completed grid merely to move points into a desired region.

This is a first reference-state diagnostic, not a completed hydraulic applicability verdict. It complements the separate spatial/timestep refinement checks.


## BASED–Manning check on actual reference sections · 29 September 2026

**Reference: HYD-2026-09-29-BASED-MANNING.** Added after deciding to retain the real-case depth/celerity methodology and measure its internal consistency before proposing changes. This supersedes any suggestion that the preceding normal-depth diagnostic should automatically replace BASED.

The sample comprises examples **03, 21, 55 and 56**, matching the earlier real pilot. We use one stored independent-experiment reference per example, not multiple copies of its parent/child states. This is a purposeful small sample, not a representative river survey.

The audit reads actual preparation manifests: BASED depth is tied to the **peak reference discharge**, and one derived $k_b=W_{ref}/D_{ref}$ is reused across states. The configured `kb_value=20` fallback in these manifests is not the resolved BASED value. Each cached reference is checked against its preparation manifest, and the installed BASED model reproduces all four stored depths. The synthetic comparison evaluates BASED at **the same 1,000 m³/s peak, 500 m width and 1e−4 slope** as its saved baseline reference. Its 100 m³/s baseflow is not used for this matched-reference test.

For each reference section:

$$U_Q=Q/(WD),\quad U_M=D^{2/3}\sqrt{S}/n,\quad Q_M=WDU_M,$$
$$r_Q=Q_M/Q=U_M/U_Q,\qquad n_{implied}=WD^{5/3}\sqrt{S}/Q=n\,r_Q.$$

$r_Q=1$ is algebraic consistency under the wide-channel Manning assumptions, not a complete hydraulic validation. $n_{implied}$ is a diagnostic value, not a calibrated or recommended replacement. The script checks raw celerity against the shared RAPID implementation; clipping is reported separately and never used to infer a physical velocity.

Rebuild the audit from the repository root with:
```bash
python synthetic_experiments/investigations/based_manning_reference_audit.py
```


In [ ]:
BASED_AUDIT=OUT/'based_reference_audit'
based_audit=pd.read_csv(BASED_AUDIT/'reference_comparison.csv',dtype={'case':str})
based_provenance=json.loads((BASED_AUDIT/'provenance.json').read_text())
print(based_provenance['reference'])
display(based_audit[['case','reference_discharge_cms','reference_width_m','reference_slope',
    'reference_depth_m','kb_value','n_manning','manning_discharge_cms',
    'q_manning_over_reference','n_implied_by_reference']].round(5))
display(based_audit[['case','velocity_from_q_mps','velocity_manning_mps','celerity_raw_mps',
    'celerity_used_mps','reference_celerity_clipped','cached_reference_matches_prep',
    'width_refresh_marker_present','width_refresh_marker_status','based_reproduction_relative_error']])
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
for ax,col,target,title,xlabel in [
    (axes[0],'q_manning_over_reference',1,'Discharge consistency','Manning-implied Q / reference Q'),
    (axes[1],'n_implied_by_reference',.35,'Roughness needed for exact local agreement','Algebraically implied Manning n')]:
    for i,row in based_audit.iterrows():
        color='#2563eb' if row.group=='Real stored BASED' else '#ea580c'
        ax.scatter(row[col],i,s=65,color=color,zorder=3)
        ax.annotate(f'{row[col]:.3f}',(row[col],i),xytext=(6,6),textcoords='offset points',fontsize=9)
    ax.axvline(target,color='#475569',ls='--',label='Exact agreement' if col=='q_manning_over_reference' else 'Configured n = 0.35')
    ax.set(xscale='log',yticks=range(len(based_audit)),yticklabels=based_audit.case,
           title=title,xlabel=xlabel,ylim=(len(based_audit)-.5,-.7))
    ax.margins(x=.2);ax.grid(axis='x',alpha=.15);ax.legend(fontsize=8,loc='lower left')
fig.suptitle('Stored real references and matched synthetic peak reference; no routing changes')
fig.savefig(BASED_AUDIT/'reference_consistency.png',dpi=180,bbox_inches='tight');display(fig);plt.close(fig)


### Findings and next decision

- **Real references:** BASED depths are 2.31–6.35 m, with resolved $k_b$ approximately 120–340. All four preparation manifests use Manning $n=0.35$. Their Manning-implied discharge is only **5.2–12.9%** of the stored reference discharge, a factor **7.8–19.2** below it. This is not a small disagreement.
- **Synthetic reference:** at the same peak discharge and geometry, BASED gives depth **3.47 m**, $k_b$ **144.3**, and $r_Q=0.113$. Its discrepancy is comparable to these real references. The saved fixed-$k_b=20$ synthetic reference instead has depth 25 m and $r_Q=3.054$ at peak flow.
- **Two separate issues:** the fixed synthetic $k_b$ differs from the real-case BASED procedure; independently, the chosen roughness and BASED references do not satisfy the same Manning discharge relation. Switching synthetic depth to BASED would align the procedure, but would not remove the latter discrepancy.
- The locally implied roughness is **0.018–0.045** for these real references and **0.0397** for the BASED synthetic reference. The next investigation is the provenance and intended meaning of **the configured 0.35**, including whether it is an effective routing parameter. These values do not prove a decimal-place error and do not justify silently changing it.

**Scope and provenance:** three selected roots (03, 21, 56) contain width-refresh-required marker files. Cached reference summaries agree with the stored prep references, but this audit does not certify that the prep represents the latest promoted branch geometry or that those markers remain actionable. It diagnoses the saved reference parameterization; final real-case validation must use refreshed, verified products. Example 55 has no such marker in the inspected root.

No stored depths, roughness values, celerity limits, forcing, or routing outputs were changed. BASED was reevaluated only for reproducibility and the synthetic diagnostic. The additional artifact hashes and source paths are recorded in `applicability/based_reference_audit/provenance.json`.


## Routing experiment: common and reference-derived Manning n

**HYD-2026-09-29-MANNING-SENSITIVITY.** Following the reference audit, test $n=0.025,0.035,0.050$ and a reference-derived value in a separate synthetic companion experiment. All seven collapse treatments are retained, including the nested main-island collapse.

Seven configurations: the baseline (500 m, slope 1e−4, A=0.8, inner=0.5, multiplier=1.25, extent=20 km, central breaks); width 250/1000 m; slope 5e−5/5e−4; and long narrow/dominant A (A=0.05/0.95, multiplier=3). This is a focused sensitivity study, not a complete factorial grid.

For each configuration, estimate one BASED depth on its single-centerline reference using total width, reference slope and the 1000 m³/s forcing peak. Compute $k_b=W_{ref}/D_{ref}$ and $n_{derived}=W_{ref}D_{ref}^{5/3}\sqrt{S_{ref}}/Q_{peak}$. **Both are shared across all parent and child states**, not recalibrated on individual branches or responses. Exact reference agreement does not guarantee agreement in every branch.

Six variants separate the interventions:

- Legacy: fixed $k_b=20$, $n=0.35$.
- Depth-only control: BASED $k_b$, $n=0.35$.
- BASED $k_b$ with $n=0.025,0.035,0.050$ or reference-derived $n$.

The 200 m segmentation/floor, width allocation, Muskingum X, forcing and celerity limits remain fixed. A shared stable timestep is recomputed across states for each configuration/variant. Thus timestep differences are reported; this is not a refinement study. The original full grid and real-case runs are unchanged.

Literature context: [USACE main-channel roughness guidance](https://www.hec.usace.army.mil/confluence/rasdocs/ras1dtechref/6.6/basic-data-requirements/geometric-data/energy-loss-coefficients) supports testing roughness in this range; these are candidate values, not independently calibrated values for our rivers.

Reproduce/resume with `python synthetic_experiments/investigations/manning_sensitivity.py`.


In [ ]:
MANNING_OUT=ROOT/'synthetic_experiments/outputs/manning_sensitivity'
manning_manifest=json.loads((MANNING_OUT/'manifest.json').read_text())
assert manning_manifest['status'] in ['complete','complete_with_infeasible'], 'Complete the focused routing experiment before interpreting these plots.'
mn=pd.read_csv(MANNING_OUT/'results.csv')
mn_refs=pd.read_csv(MANNING_OUT/'references.csv')
mn_infeasible=pd.read_csv(MANNING_OUT/'infeasible.csv')
display(mn_infeasible)
print('Attempted comparisons:',manning_manifest['attempted_comparisons'],'; not routed under shared-clock constraint:',manning_manifest['infeasible_comparisons'])
MN_Y=['lag_shift_pct','event_duration_change_pct','peak_excess_change_pct']
MN_LABELS=['Peak-lag change','Event-duration change','Peak-excess change']
MN_STATES=['simple/full','nested/inner_only','nested/main_island_only','nested/full','crossover/upstream_only','crossover/downstream_only','crossover/full']
MN_VARIANTS=['based_n0025','based_n0035','based_derived','based_n0050']
print(f"{len(mn)} comparisons; {manning_manifest['unique_runs']} unique routes; {manning_manifest['legacy_original_grid_matches']} legacy comparisons reproduce the original grid.")
display(mn_refs.round(5))
mn_qa=mn.assign(capped=mn.parent_n_celerity_capped+mn.child_n_celerity_capped,
    lower_clipped=mn.parent_n_lower_celerity_clipped+mn.child_n_lower_celerity_clipped,
    upper_clipped=mn.parent_n_upper_celerity_clipped+mn.child_n_upper_celerity_clipped,
    floored=mn.parent_n_length_floored+mn.child_n_length_floored)
display(mn_qa.groupby('variant').agg(comparisons=('case','size'),dt_min=('dt_seconds','min'),dt_max=('dt_seconds','max'),
    pairs_with_clipping=('capped',lambda x:int((x>0).sum())),pairs_with_lower_clipping=('lower_clipped',lambda x:int((x>0).sum())),
    pairs_with_upper_clipping=('upper_clipped',lambda x:int((x>0).sum())),pairs_with_floor=('floored',lambda x:int((x>0).sum())),
    lag_within_one_dt=('within_one_dt','sum'),max_volume_error=('max_volume_error','max')))
# Each pair retains identical geometry/omega across parameter variants.
assert mn.groupby(['case','child_state']).resistance_predictor.apply(lambda x:np.allclose(x,x.iloc[0])).all()


### Baseline responses: change roughness while retaining BASED depth

Each line is one collapse comparison; points include the derived reference n (diamond). Lines connect tested levels, without implying a fitted response curve. The table underneath separates legacy, depth-only and roughness interventions. These are child/parent percentages, not absolute travel times.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,5),constrained_layout=True)
for color,state in zip(plt.get_cmap('tab10').colors,MN_STATES):
    g=mn[(mn.case=='baseline')&(mn.child_state==state)&mn.variant.isin(MN_VARIANTS)].sort_values('n_manning')
    derived=g[g.variant=='based_derived']
    for ax,y,label in zip(axes,MN_Y,MN_LABELS):
        ax.plot(g.n_manning,g[y],marker='o',color=color,label=state)
        ax.scatter(derived.n_manning,derived[y],marker='D',s=45,color=color)
        ax.set(xlabel='Manning n',ylabel='Child/parent change (%)',title=label)
        ax.grid(alpha=.15)
fig.legend(*axes[0].get_legend_handles_labels(),loc='outside lower center',ncol=4,fontsize=8)
fig.suptitle('Baseline geometry; common BASED kb; diamond = reference-derived n')
fig.savefig(MANNING_OUT/'baseline_responses.png',dpi=180,bbox_inches='tight');display(fig);plt.close(fig)
display(mn[mn.case=='baseline'].pivot(index='child_state',columns='variant',values=MN_Y).round(2))


### How much do percentages change across the tested roughness values?

These panels show the **maximum minus minimum response** across 0.025, 0.035, 0.050 and the derived n, at fixed geometry and BASED kb. Units are percentage points, not variance contributions. Larger ranges identify comparisons worth a closer hydrograph/resolution check. The common derived n differs between reference configurations and is shown in the reference table.


Missing cells denote configurations without enough feasible tested roughness values, not zero sensitivity. The coverage table gives the number of completed values. Failure means no integer timestep satisfies the shared all-state Muskingum bounds under the current segmentation, floor and X; it is not evidence of physical impossibility.

In [ ]:
mn_range=mn[mn.variant.isin(MN_VARIANTS)].groupby(['case','child_state'])[MN_Y].agg(lambda x:x.max()-x.min())
mn_counts=mn[mn.variant.isin(MN_VARIANTS)].groupby(['case','child_state']).size()
mn_range.loc[mn_counts<2,:]=np.nan
mn_range.to_csv(MANNING_OUT/'roughness_response_ranges.csv')
display(mn_counts.unstack().reindex(index=mn_refs.case,columns=MN_STATES).fillna(0).astype(int))
fig,axes=plt.subplots(1,3,figsize=(17,6),constrained_layout=True)
for ax,y,label in zip(axes,MN_Y,MN_LABELS):
    table=mn_range[y].unstack().reindex(index=mn_refs.case,columns=MN_STATES)
    im=ax.imshow(table,aspect='auto',cmap='YlOrRd',vmin=0)
    ax.set_xticks(range(7),MN_STATES,rotation=55,ha='right',fontsize=8)
    ax.set_yticks(range(7),table.index,fontsize=9);ax.set_title(label)
    for i in range(7):
        for j in range(7):
            value=table.iloc[i,j]
            ax.text(j,i,f'{value:.1f}' if np.isfinite(value) else '—',ha='center',va='center',fontsize=8,color='white' if value>table.max().max()*.6 else '#222')
    fig.colorbar(im,ax=ax,shrink=.65,label='Range (percentage points)')
fig.savefig(MANNING_OUT/'roughness_response_ranges.png',dpi=180,bbox_inches='tight');display(fig);plt.close(fig)
display(mn_range.agg(['median','max']).round(3))


### Baseline nested collapse: inspect the hydrographs

Nested/full at the baseline geometry (outer A=0.8, inner=0.5, selected branch multiplier=1.25). Parent and child use the same reference parameters within each panel. Time is aligned to forcing onset using each run's saved spin-up duration; discharge is excess over the common initial plateau. Plots include the entire event above the 0.5% parent-amplitude threshold plus a margin. The three response metrics remain authoritative; visually apparent peaks are not automatically classified as distinct events.


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(13,8),constrained_layout=True)
for ax,variant in zip(axes.flat,MN_VARIANTS):
    row=mn[(mn.case=='baseline')&(mn.child_state=='nested/full')&(mn.variant==variant)].iloc[0]
    end=0
    for side,color in [('parent','#2563eb'),('child','#ea580c')]:
        folder=MANNING_OUT/'runs'/row[side+'_run_id']
        hydro=pd.read_parquet(folder/'hydrograph.parquet');mt=json.loads((folder/'metrics.json').read_text())
        time=(hydro.time_seconds.to_numpy()+mt['dt_seconds']/2-mt['buffer_hours']*3600)/3600
        excess=hydro.outlet_cms.to_numpy()-100
        ax.plot(time,excess,color=color,label=side)
        event=(time>=0)&(excess>.005*(row.parent_peak_cms-100))
        if event.any():end=max(end,time[event].max())
    ax.set(xlim=(0,end+5),xlabel='Time since forcing onset (hours)',ylabel='Excess discharge (m³/s)',
        title=f"{'Derived' if variant=='based_derived' else 'Fixed'} n={row.n_manning:.4f}; dt={row.dt_seconds:g} s")
    ax.grid(alpha=.15);ax.legend()
fig.savefig(MANNING_OUT/'baseline_nested_hydrographs.png',dpi=180);display(fig);plt.close(fig)


### Roughness experiment: findings and limits

**Completed:** 266 parent–child comparisons from 304 unique routing runs. Of 294 planned comparisons, 28 were not routed: all four new roughness variants in the long-narrow-A configuration have no shared stable all-state timestep under the retained segmentation/floor and X=0.1. No settings were altered to force those runs. This does not imply every individual treatment is infeasible under a different clock design; it means the common-clock experiment cannot run that configuration as specified.

For the 42 geometry/treatment combinations with all four new roughness variants, the median response range is **0.30 percentage points for lag**, **4.07 for duration**, and **2.91 for peak excess**; maxima are 0.90, 17.42 and 19.44 points. These ranges include the configuration-derived n, which is approximately **0.100** for the 1000 m reference at the unchanged 1000 m³/s peak. It is outside the fixed 0.025–0.050 test interval, showing why a consistency-derived n is not automatically a physically preferred roughness.

In the baseline simple/full case, n=0.025 → 0.050 changes lag from −27.08% to −26.97%, duration from −3.80% to −9.63%, and peak excess from +3.68% to +7.77%. Faster routing relative to the fixed forcing duration is consistent with reduced collapse-related shape changes here. This does not establish the same direction for every geometry or a universal scaling law.

The geometry and omega are invariant across these variants. Thus shape changes can vary substantially at the same omega; n and the BASED reference affect routing timescales rather than geometric predictor values. BASED-depth-only and legacy controls remain visible to prevent conflating depth and roughness interventions.

**Checks:** all completed runs passed volume and plateau checks (maximum relative event-volume error <5e−14); 35 legacy comparisons matched the original grid's three responses. Shared-clock timestep changes and celerity clipping remain possible contributors; this is not a convergence demonstration. The long-narrow failure requires a separate, explicitly designed numerical follow-up before extending these conclusions to that case. No real runs or original-grid outputs were changed.


## Segmentation test · 29 September 2026

**HYD-2026-09-29-SEGMENTATION.** Test whether shorter routing segments resolve the long-narrow case, and distinguish changes in timing from changes in shape. This is a numerical-method sensitivity study; none of these alternatives has been adopted as the main routing policy.

Three configurations (baseline, long-narrow A, long-dominant A), all seven comparisons, n=0.035 and the common BASED kb≈144.3. Width, slope, physical lengths, flow allocation, forcing, X=0.1 and celerity limits are unchanged.

- **Uniform 200/100/50 m:** use the corresponding effective-length floor, as in the existing refinement audit.
- **Adaptive 200/100/50 m maximum:** retain original physical geometry, subdivide slower edges further to satisfy the shared timestep bounds; disable the artificial effective-length floor for these computational subdivisions.
- **Real-workflow safeguard:** apply the actual `apply_stability_effective_length_cap` helper with its 200 m floor and 10 s safety margin to the uniform 200 m frames. This tests the helper on these synthetic geometries, not every step of real-case preprocessing.
- **Timestep-only controls:** where possible, repeat each adaptive mesh at its smallest admissible integer timestep. The main adaptive clock is 80% of the uniform mesh's upper bound.

The shared routing solver, forcing and QA are reused with an isolated preparation binding; production source/defaults are not modified. Three baseline states are checked against the stock adapter at the same clock. Run/resume using `python synthetic_experiments/investigations/segmentation_sensitivity.py`.


In [ ]:
SEG_OUT=ROOT/'synthetic_experiments/outputs/segmentation_sensitivity'
seg_manifest=json.loads((SEG_OUT/'manifest.json').read_text())
assert seg_manifest['status']=='complete'
seg=pd.read_csv(SEG_OUT/'comparisons.csv')
seg_runs=pd.read_csv(SEG_OUT/'run_diagnostics.csv')
seg_failed=pd.read_csv(SEG_OUT/'infeasible.csv')
SEG_CASES=['baseline','long_narrow','long_dominant']
SEG_VARIANTS=['adaptive_200','adaptive_100','adaptive_50']
print(f"Completed {len(seg)} comparisons; {seg_manifest['unique_runs']} unique routes. Infeasible variants are shown below.")
display(seg_failed.round(3))
display(seg_runs.groupby(['case','variant']).agg(states=('state','size'),dt_seconds=('dt_seconds','first'),
    min_segment_m=('min_segment_length_m','min'),max_segment_m=('max_segment_length_m','max'),
    max_edge_k_relative_change=('max_edge_k_relative_change','max'),max_volume_error=('volume_error_fraction',lambda x:x.abs().max())))
assert np.allclose(seg_runs.centroid_lag_seconds,seg_runs.predicted_discrete_centroid_lag_seconds,rtol=1e-7,atol=.01)
assert np.allclose(seg_runs.added_variance_seconds2,seg_runs.predicted_discrete_added_variance_seconds2,rtol=1e-7,atol=.1)


### Why mean travel time and spreading respond differently

For a fixed path with constant-coefficient Muskingum elements, the transfer-function moments give

$$\mu_{path}=\sum_iK_i,\qquad \sigma^2_{path}=(1-2X)\sum_iK_i^2.$$

Splitting one element into $m$ equal elements preserves $\sum K_i$ but reduces its variance contribution by a factor $m$. Across parallel paths, another contribution is the variance of the width-allocation-weighted path means. That **between-path contribution remains** when path travel times and allocations are preserved. Refinement reduces within-path spreading; it does not erase differences in branch arrival times.

These are model moments, not proof of physical dispersion. At fixed X, progressively finer constant-property segmentation tends toward zero within-path Muskingum spreading. Therefore a refinement sequence is not automatically convergence to a physically validated flood-wave model. [USACE explicitly notes the attenuation dependence on subreach count](https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/applying-reach-routing-methods-within-hec-hms/applying-the-muskingum-routing-method).

The timing centroid measures the mean of excess event volume, whereas **peak time is not a mean**. Altered spreading can move the peak or change which peak is largest. Our primary response remains peak lag; centroid lag is an explanatory diagnostic only.

**Implementation check:** RAPID's interval-mean external boundary treatment contributes known moment corrections. For these single-inlet networks, predicted measured centroid lag is $\sum a_p\mu_p-XK_{inlet}-\Delta t/2$. Added output variance is the within-plus-between contribution plus $(XK_{inlet})^2-\Delta t^2/4$. These corrected expressions reproduce the measured moments. Thus the small centroid shifts with timestep/mesh should not be described as physical changes in branch travel time.


In [ ]:
seg_simple=seg_runs[(seg_runs.state=='simple/original')&seg_runs.variant.isin(SEG_VARIANTS)].copy()
seg_simple['path_mean_hours']=seg_simple.path_mean_k_seconds/3600
seg_simple['centroid_lag_hours']=seg_simple.centroid_lag_seconds/3600
seg_simple['peak_lag_hours']=seg_simple.peak_lag_seconds/3600
seg_simple['within_path_sd_hours']=np.sqrt(seg_simple.within_path_variance_seconds2)/3600
seg_simple['between_path_sd_hours']=np.sqrt(seg_simple.between_path_variance_seconds2)/3600
display(seg_simple[['case','variant','n_segments','dt_seconds','path_mean_hours','centroid_lag_hours','peak_lag_hours',
    'peak_cms','within_path_sd_hours','between_path_sd_hours']].round(4))
for _,g in seg_simple.groupby('case'):
    assert np.allclose(g.path_mean_k_seconds,g.path_mean_k_seconds.iloc[0])
    assert np.allclose(g.between_path_variance_seconds2,g.between_path_variance_seconds2.iloc[0])


### Effect on our three primary parent–child responses

Signed difference: adaptive 50 m maximum minus adaptive 200 m maximum, in **percentage points**. Both runs use the same geometry and reference hydraulics. This includes the associated change in admissible timestep; timestep-only comparisons are tabulated separately. The long-narrow case has no valid uniform-200 m comparator, so these cells compare two adaptive choices rather than claiming a correction relative to a runnable original mesh.


In [ ]:
seg_key=['case','child_state']
seg_delta=seg[seg.variant=='adaptive_50'].set_index(seg_key)[MN_Y]-seg[seg.variant=='adaptive_200'].set_index(seg_key)[MN_Y]
seg_delta.to_csv(SEG_OUT/'adaptive_50_minus_200_pp.csv')
fig,axes=plt.subplots(1,3,figsize=(14,6),constrained_layout=True)
for ax,y,label in zip(axes,MN_Y,MN_LABELS):
    table=seg_delta[y].unstack('case').reindex(index=MN_STATES,columns=SEG_CASES)
    limit=max(abs(table.to_numpy()).max(),.01)
    im=ax.imshow(table,aspect='auto',cmap='RdBu',vmin=-limit,vmax=limit)
    ax.set_xticks(range(3),SEG_CASES,rotation=35,ha='right');ax.set_yticks(range(7),MN_STATES,fontsize=8);ax.set_title(label)
    for i in range(7):
        for j in range(3):
            value=table.iloc[i,j]
            ax.text(j,i,f'{value:+.2f}',ha='center',va='center',fontsize=8,color='white' if abs(value)>.65*limit else '#222')
    fig.colorbar(im,ax=ax,shrink=.7,label='Difference (percentage points)')
fig.savefig(SEG_OUT/'response_refinement_changes.png',dpi=180);display(fig);plt.close(fig)
seg_clock=[]
for label in SEG_VARIANTS:
    a=seg[seg.variant==label].set_index(seg_key)[MN_Y]
    b=seg[seg.variant==label+'_small_dt'].set_index(seg_key)[MN_Y]
    delta=(b-a).dropna()
    for case,g in delta.groupby(level='case'):
        seg_clock.append(dict(case=case,mesh=label,**{y:float(g[y].abs().max()) for y in MN_Y}))
display(pd.DataFrame(seg_clock).rename(columns={y:'max_abs_delta_pp_'+y for y in MN_Y}).round(4))


### Parent hydrographs: the same branches, different routing subdivision

Simple parents are shown at the three adaptive resolutions. Excess discharge is measured above the common plateau. The long-narrow panel retains the late secondary arrival; its timescale differs from the other panels. These are absolute parent hydrographs, complementing the child/parent response differences above.


In [ ]:
fig,axes=plt.subplots(3,1,figsize=(12,10),constrained_layout=True)
for ax,case in zip(axes,SEG_CASES):
    end=0
    for variant,color,style in zip(SEG_VARIANTS,['#2563eb','#ea580c','#16a34a'],['-','--',':']):
        row=seg_runs[(seg_runs.case==case)&(seg_runs.state=='simple/original')&(seg_runs.variant==variant)].iloc[0]
        h=pd.read_parquet(SEG_OUT/'runs'/row.run_id/'hydrograph.parquet')
        t=(h.time_seconds.to_numpy()+row.dt_seconds/2-row.buffer_hours*3600)/3600
        q=h.outlet_cms.to_numpy()-100
        ax.plot(t,q,color=color,ls=style,label=variant.replace('adaptive_','Maximum ')+ ' m')
        mask=(t>=0)&(q>.005*(row.peak_cms-100))
        end=max(end,t[mask].max())
    ax.set(xlim=(0,end+5),xlabel='Time since forcing onset (hours)',ylabel='Excess discharge (m³/s)',title=case)
    ax.grid(alpha=.15);ax.legend(fontsize=9)
fig.savefig(SEG_OUT/'parent_refinement_hydrographs.png',dpi=180);display(fig);plt.close(fig)


### Segmentation findings and decision boundary

147 comparisons completed from 128 unique routes. The long-narrow configuration remains infeasible under uniform 200, 100 and 50 m segmentation and under the real-workflow helper with the 200 m floor. Adaptive subdivision succeeds, with shortest segments approximately 71, 36 and 18 m for maximum targets 200, 100 and 50 m respectively.

Adaptive 200 → 50 m maximum produces these largest absolute differences across the seven treatments (percentage points):

| Configuration | Peak-lag change | Event-duration change | Peak-excess change |
|---|---:|---:|---:|
| Baseline | 4.56 | 0.59 | 0.39 |
| Long narrow A | 0.37 | 0.22 | 0.20 |
| Long dominant A | 1.54 | 2.46 | 1.32 |

**Do not summarize this as “shape changes but timing does not.”** Width-weighted path mean travel time is preserved by physical subdivision, while peak timing can move as spreading changes. In the baseline nested/main-island comparison, peak-lag change moves by 4.56 pp despite very small duration/peak-excess changes. For baseline simple/full, the parent peak arrives about 14.3 minutes earlier on the finer mesh while its path-mean travel time remains constant. Conversely, long-dominant simple/full is more sensitive in duration and peak excess than in relative peak lag.

At the same baseline adaptive-200 mesh, reducing only the timestep from 264 s to 108 s changes lag response by at most 0.59 pp, duration by 0.013 pp and peak excess by 0.012 pp. This separates an output-clock contribution from the broader subdivision dependence; it does not fully isolate spatial effects across all meshes because their admissible clocks differ.

All completed runs pass conservation and plateau checks (maximum relative event-volume error below 2e−13). Adapter parity was verified against stock routing. Analytical within/between-path moments with the external-boundary correction reproduce measured moments to floating-point precision. These checks validate the implementation of this experiment, not hydrodynamic realism.

**Recommendation:** retain the 200 m convention as the main-analysis candidate while assessing its sensitivity explicitly. Treat adaptive long-narrow results as a separate numerical-method result for now. Its response percentages are close across the tested adaptive meshes, but the mesh rule changes attenuation and should not be silently mixed into the original grid or real-case comparison. No production policy or full rerun has been changed by this test.


## Exact edge controllers · HYD-2026-09-29-EDGE-CONTROLLERS

Trace the timestep bounds to physical branches before proposing another parameter change. Scope: the seven focused Manning-test configurations, at n=0.035, their saved BASED kb, 200 m segmentation/floor and X=0.1. This is not yet the full-grid feasibility scan.

The long-narrow example alone fails among these seven. The common upper bound is set by the 500 m approach/exit channels: c=1.0904 m/s, segment K=183.4 s, maximum dt=330.2 s. The nested upper-inner branch and crossover edge at x=8–12 km are only 12.5 m wide, physically 12 km long, with slope 3.333e−5. Their implied depth is 0.0866 m, c=0.05383 m/s, segment K=3715.7 s, and minimum dt=743.1 s. Neither controller is clipped at this parameterization.

The unlengthened 12.5 m inner/continuation branches also demand dt≥429.1 s, so removing the threefold lengthening does not remove all offending branches. The simple narrow A is 25 m wide, physically 30 km long, c=0.08544 m/s and requires dt≥468.2 s. All computational segments are 200 m: longer physical branches affect their segment celerity through the reduced head gradient, not by remaining long unsplit computational reaches.

At equal segment lengths, common n/kb and no clipping, the ratio against the full-width reference channel is

$$c_{main}/c_e=(W_{main}/W_e)^{2/3}\sqrt{S_{main}/S_e}.$$

With X=0.1 the admissible ratio is at most nine. For Wmain=500 m this gives a continuous threshold width of 18.52 m at equal slope, or 42.21 m at one-third slope. A 50/50 inner split therefore needs outer A of approximately 16.9% or more for its threefold-lengthened branch to meet this particular constraint. This threshold is conditional on these exact assumptions; it is not a universal river-width requirement, and integer timestep selection needs margin.

These branches carry only 2.5% of inlet flow under the allocation rule, but the global nonnegative-coefficient bound is not discharge-weighted. The direct structural issue is the contrast between a full-width main channel and very narrow, shallow, low-slope internal branches. The imposed depth scaling D=W/kb down to those branches deserves physical scrutiny; changing the common kb does not remove the uncapped contrast.

Reproduce with `python synthetic_experiments/investigations/stability_edge_audit.py`. All canonical edges, controller flags and state bounds are saved; original edge labels are retained where a canonical reach has a direct source-edge match.


In [ ]:
EDGE_OUT=SEG_OUT/'edge_audit'
edge_audit=pd.read_csv(EDGE_OUT/'edges.csv')
display(pd.read_csv(EDGE_OUT/'cases.csv').round(3))
edge_show=edge_audit[(edge_audit.case=='long_narrow_A')&edge_audit.state.isin(['simple/original','nested/original','crossover/original'])&(edge_audit.blocks_case_fast_bound|edge_audit.controls_case_upper)]
display(edge_show[['state','edge_label','start_km','end_km','width_m','length_m','slope','depth_m',
    'segment_length_m','celerity_raw_mps','celerity_used_mps','segment_K_seconds',
    'dt_lower_seconds','dt_upper_seconds','clipped','flow_share']].round(5))
display(pd.read_csv(EDGE_OUT/'states.csv').query("case=='long_narrow_A'").round(3))


## BASED-anchored width–depth scaling · 29 September 2026

**HYD-2026-09-29-WIDTH-DEPTH.** Requested to test whether narrower branches can have lower width/depth ratios while keeping the existing BASED reference. The diagnostic now includes the selected synthetic α=0.65; shared production defaults and completed main-grid outputs remain unchanged.

\[
D_i=D_{\rm ref}(W_i/W_{\rm ref})^\alpha,\qquad
k_{b,i}=k_{b,\rm ref}(W_i/W_{\rm ref})^{1-\alpha}.
\]

Test **α = 1, 0.8, 0.71, 0.65, 0.6, 0.5**. α=1 exactly recovers constant kb; 0.5 represents stronger moderation. [Li & Limaye (2025), equation 9](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2024JF008196) use a depth–width exponent of 0.71 for braided channel threads in a migration model. Our BASED normalization and its use for routing are additional assumptions, not validated by that study.

**Controls:** n=0.035; X=0.1; 200 m target and effective-length floor; original forcing, widths, length/head-loss construction and width-based allocation. One reference section is shared across all states of a configuration. The local kb is also passed into the existing Chezy resistance solver; Cz remains constant, so this remains the Chezy predictor rather than a Manning-equivalent resistance.

**Scope:** constraint preflight of 2,187 original grid configurations and 140 embedded-extension configurations; actual routing of seven focused geometries × six exponents × seven collapse comparisons where feasible. Real checks cover stored base networks 03/21/55/56 under the nominal settings only, with no real hydrograph reruns or all-state guarantee. See width-refresh markers below.

Reproduce with `investigations/width_depth_scaling.py`; separate outputs are in `outputs/width_depth_scaling/`.

In [ ]:
WD_OUT=ROOT/'synthetic_experiments/outputs/width_depth_scaling'
wd_manifest=json.loads((WD_OUT/'manifest.json').read_text())
assert wd_manifest['status'].startswith('complete')
wd=pd.read_csv(WD_OUT/'results.csv')
wd_states=pd.read_csv(WD_OUT/'states.csv')
wd_edges=pd.read_csv(WD_OUT/'edges.csv')
wd_grid=pd.read_csv(WD_OUT/'grid_preflight.csv')
wd_real=pd.read_csv(WD_OUT/'real_base_preflight.csv',dtype={'example':str})
wd_bounds=pd.read_csv(WD_OUT/'bounds.csv')
WD_Y=['lag_shift_pct','event_duration_change_pct','peak_excess_change_pct']
WD_LABELS=['Lag shift (%)','Event-duration change (%)','Peak-excess change (%)']
WD_ALPHA=wd_manifest['alphas']
print({k:wd_manifest[k] for k in ['native_comparisons','comparisons','unique_runs','previous_comparisons_reproduced','parity_hydrographs','max_volume_error']})
display(wd_grid.groupby(['dataset','alpha']).feasible.agg(passing='sum',tested='count'))
display(wd_real[['example','alpha','feasible','dt_lower_seconds','dt_upper_seconds','width_refresh_marker']].round(2))
print('Focused all-state bounds:')
display(wd_bounds.round(2))
fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)
ref=pd.read_csv(ROOT/'synthetic_experiments/outputs/manning_sensitivity/references.csv').set_index('case').loc['baseline']
w=np.geomspace(10,1500,200)
for alpha in WD_ALPHA:
    depth=ref.based_depth_m*(w/ref.width_m)**alpha
    axes[0].plot(w,depth,label=f'α={alpha:g}')
    axes[1].plot(w,w/depth,label=f'α={alpha:g}')
for ax in axes:
    ax.set(xscale='log',yscale='log',xlabel='Branch width (m)')
    ax.axvline(ref.width_m,color='grey',ls=':',lw=1)
    ax.grid(alpha=.15)
axes[0].scatter([ref.width_m],[ref.based_depth_m],c='black',zorder=5)
axes[1].scatter([ref.width_m],[ref.kb],c='black',zorder=5)
axes[0].set(ylabel='Assigned depth (m)',title='Same BASED anchor; moderated depth contrast')
axes[1].set(ylabel='Local width/depth ratio kb',title='Narrow branches receive lower kb')
axes[1].legend()
display(fig);plt.close(fig)

### Response changes with an identical timestep

The panels below use a **common clock across feasible exponents within each geometry**. This controls the timestep and forcing-sampling phase while changing the hydraulic assumption. Parent and child always use the same settings. Each line is one parent–child collapse comparison; α decreases from left to right toward stronger moderation.

For long-narrow A, α=0.65, 0.6 and 0.5 are feasible among these six choices. Its short three-point curves cover only that range; the missing points are infeasible, not missing zero responses. The complete result table also contains the native, separately selected clocks. The following table measures the maximum native-versus-common-clock response difference for each geometry.

In [ ]:
wd_common=wd[wd.clock_mode=='common_clock'].copy()
wd_native=wd[wd.clock_mode=='native_clock'].copy()
wd_clock=wd_native.merge(wd_common,on=['case','alpha','child_state'],suffixes=('_native','_common'),validate='one_to_one')
for y in WD_Y:wd_clock[y]=abs(wd_clock[y+'_native']-wd_clock[y+'_common'])
display(wd_clock.groupby('case')[WD_Y].max().rename(columns=lambda c:c+' | max clock difference (pp)').round(3))
WD_CASES=['baseline','long_narrow_A','long_dominant_A']
WD_TREATMENTS=sorted(wd.child_state.unique())
wd_colors=dict(zip(WD_TREATMENTS,plt.get_cmap('tab10').colors))
fig,axes=plt.subplots(3,3,figsize=(16,11),sharex=True,constrained_layout=True)
for col,case in enumerate(WD_CASES):
    for row,(y,label) in enumerate(zip(WD_Y,WD_LABELS)):
        ax=axes[row,col]
        for treatment in WD_TREATMENTS:
            d=wd_common[(wd_common.case==case)&(wd_common.child_state==treatment)].sort_values('alpha',ascending=False)
            ax.plot(d.alpha,d[y],'-o',ms=4,color=wd_colors[treatment],label=treatment)
        ax.set_xlim(1.03,.47);ax.set_xticks(WD_ALPHA)
        ax.grid(alpha=.15)
        if col==0:ax.set_ylabel(label)
        if row==0:ax.set_title(case.replace('_',' '))
        if row==2:ax.set_xlabel('Depth exponent α (stronger moderation →)')
handles,labels=axes[0,0].get_legend_handles_labels()
fig.legend(handles,labels,loc='outside lower center',ncol=4,fontsize=9)
display(fig);plt.close(fig)
wd_delta=wd_common[wd_common.alpha==.5].set_index(['case','child_state'])[WD_Y]-wd_common[wd_common.alpha==1].set_index(['case','child_state'])[WD_Y]
print('Maximum absolute α=0.5 minus α=1 response change across treatments (percentage points):')
display(wd_delta.abs().groupby('case').max().round(3))

### Recomputed resistance and reference-flow consistency

Resistance is recalculated using each branch's local kb. Length fraction and width-derived exposure retain their geometric definitions. The table keeps these predictors beside the three response variables; it does not fit a new predictor model.

For the narrow nested branch, also inspect **Manning-implied Q / allocated reference Q**. Allocated Q is the steady width allocation of the 1,000 m³/s inlet reference, not the routed transient peak. Improving the coefficient bounds does not by itself make those two discharge estimates agree.

In [ ]:
display(wd_common[(wd_common.case=='baseline')&(wd_common.alpha.isin([1,.71,.65,.6,.5]))][['child_state','alpha','Lambda_matched','length_fraction','width_fraction','resistance_predictor']+WD_Y].sort_values(['child_state','alpha'],ascending=[True,False]).round(4))
# Select the controlling long 12.5m branch explicitly so alpha remains visible.
critical=wd_edges[(wd_edges.case=='long_narrow_A')&(wd_edges.state=='nested/original')&(np.isclose(wd_edges.width,12.5))&(wd_edges.slope_used<1e-4)].sort_values('alpha',ascending=False)
display(critical[['alpha','width','diagnostic_depth_m','rapid_kb_value','slope_used','rapid_celerity_mps','reference_q_allocated_cms','manning_implied_q_cms','qm_over_allocated','velocity_allocated_mps']].round(4))
fig,axes=plt.subplots(1,3,figsize=(16,4),constrained_layout=True)
for ax,case in zip(axes,WD_CASES):
    rows=wd_common[(wd_common.case==case)&(wd_common.child_state=='simple/full')]
    for _,r in rows.sort_values('alpha',ascending=False).iterrows():
        h=pd.read_parquet(WD_OUT/'runs'/r.parent_run_id/'hydrograph.parquet')
        job=json.loads((WD_OUT/'runs'/r.parent_run_id/'job.json').read_text())
        clock=job['clock'];dt=clock['dt_seconds'];onset=np.ceil(clock['buffer_hours']*3600/dt)*dt
        t=(h.time_seconds-onset+dt/2)/3600
        ax.plot(t,h.outlet_cms-100,label=f'Parent α={r.alpha:g}')
    r=rows.iloc[-1]
    h=pd.read_parquet(WD_OUT/'runs'/r.child_run_id/'hydrograph.parquet')
    job=json.loads((WD_OUT/'runs'/r.child_run_id/'job.json').read_text());clock=job['clock'];dt=clock['dt_seconds'];onset=np.ceil(clock['buffer_hours']*3600/dt)*dt
    t=(h.time_seconds-onset+dt/2)/3600
    ax.plot(t,h.outlet_cms-100,'k--',label='Collapsed centerline')
    ax.set(xlim=(0,70),xlabel='Time from event onset (h)',title=case.replace('_',' '));ax.grid(alpha=.15)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Discharge above baseflow (m³/s)')
display(fig);plt.close(fig)

### Decision boundary

- **Feasibility is conditional:** all 2,187 original-grid configurations pass at every exponent under these settings; the embedded extension passes 108/140 at α=1, 132/140 at 0.8, 136/140 at 0.71 and 140/140 at both 0.6 and 0.5. These are preflight results, not full-grid hydrograph reruns.
- **Real networks are not universally resolved:** under the nominal base-only audit, α=0.6 and 0.5 resolve example 56; 21 and 55 remain infeasible. Stored geometry freshness and the actual preparation safeguards require a separate audit before real reruns.
- **Physical depth remains uncertain:** α=0.5 is a useful sensitivity scenario, not a calibrated choice or a result justified merely by passing the numerical bounds. Examine allocated-flow consistency as well as depth and compare any available bathymetry before selecting a default.
- **Hydrograph and predictor results depend on this assumption:** all three responses and local-kb resistance are recomputed. No production defaults, existing grid outputs, or real routing products were changed.

The reference is retained in [the dated synthetic checkpoint](../notes/2026-09-29_synthetic_checkpoint.md).

**Completed results:** 273 primary comparisons plus 273 common-clock controls, from 568 unique routes. Three stock-adapter hydrographs match, and all 42 available α=1 comparisons reproduce the previous BASED/n=0.035 results. Maximum relative volume error is 6.7×10⁻¹⁴; all routed cases pass plateau checks.

At baseline simple/full, α=1 → 0.5 changes lag shift −27.08% → −21.52%, duration change −6.20% → −0.67%, and peak-excess change +5.27% → +0.69%. Across baseline treatments the largest changes are 13.33, 5.53 and 4.57 percentage points respectively. The largest native/common-clock differences across this experiment are only 0.62, 0.028 and 0.032 pp, so those larger responses are not explained by clock selection.

The critical long-narrow 12.5 m branch deepens from 0.087 to 0.548 m, but its Manning-implied reference discharge rises only from 0.035 to 0.756 m³/s against the prescribed steady width allocation of 25 m³/s. Thus this scaling improves depth plausibility and numerical feasibility without resolving the discharge-allocation/hydraulic-closure mismatch. The allocated-flow velocity (23.1 → 3.65 m/s) is a reference diagnostic, not a simulated local velocity.

**Working conclusion:** retain this as a meaningful depth-assumption sensitivity. Do not select α=0.5 solely because it passes; resolve the remaining physical reference-flow interpretation before freezing a main hydraulic parameterization.


## Which configurations fail? · α=0.6 extension and real-case evidence

**HYD-2026-09-29-ALPHA06-FAILURE-REVIEW.** Added at the user's request to distinguish extreme synthetic setups from broader applicability constraints. No networks were removed and no exponent was adopted.

The units of counting differ:

- **Configuration:** one set of design parameters; its shared clock must accommodate all 11 state references.
- **Unique network:** canonical directed geometry plus the BASED reference kb, deduplicated within a dataset, without including the selected clock. The extension has 603 unique networks across 1,540 state references.
- **Comparison:** a parent–child pair requiring a common clock for those two states. A failing configuration need not invalidate every pair in it.

A failure here means no integer timestep satisfies the nonnegative-coefficient bounds at the tested settings. This is a preflight diagnostic, not an observed blow-up. The original grid and extension remain separate; these counts are not new completed hydrograph runs.

**Subsequent decision, 30 September:** α=0.65 is now selected for synthetics. Real-case selection remains deferred pending the new slope-based runs. The preceding candidate discussion records the earlier diagnostic stage.

In [ ]:
WD_SUMMARY=pd.read_csv(WD_OUT/'failure_summary.csv')
WD_PREFLIGHT_STATES=pd.read_csv(WD_OUT/'state_preflight.csv')
WD_PREFLIGHT_PAIRS=pd.read_csv(WD_OUT/'pair_preflight.csv')
wd_fail_params=pd.read_csv(WD_OUT/'failed_parameter_combinations.csv')
display(WD_SUMMARY.sort_values(['dataset','alpha'],ascending=[True,False]))
print('Exact failing extension parameter combinations (each occurs at four extents):')
display(wd_fail_params.sort_values(['alpha','outer_a_fraction','length_multiplier'],ascending=[False,True,True]))
print('Failed state types:')
display(pd.read_csv(WD_OUT/'failed_states_by_type.csv'))
fig,axes=plt.subplots(1,len(WD_ALPHA),figsize=(21,4),constrained_layout=True)
a_values=sorted(WD_PREFLIGHT_STATES.outer_a_fraction.unique())
subset=WD_PREFLIGHT_STATES[(WD_PREFLIGHT_STATES.dataset=='embedded_width_length_extension')&(WD_PREFLIGHT_STATES.embedded_extent_m==20000)]
a_values=sorted(subset.outer_a_fraction.unique());m_values=sorted(subset.length_multiplier.unique())
for ax,alpha in zip(axes,WD_ALPHA):
    count=subset[subset.alpha==alpha].groupby(['outer_a_fraction','length_multiplier']).feasible.agg(lambda v:(~v).sum()).unstack().reindex(index=a_values,columns=m_values)
    im=ax.imshow(count,origin='lower',vmin=0,vmax=5,cmap='YlOrRd',aspect='auto')
    for i in range(len(a_values)):
        for j in range(len(m_values)):ax.text(j,i,str(int(count.iloc[i,j])),ha='center',va='center',fontsize=9)
    ax.set(xticks=range(len(m_values)),xticklabels=m_values,yticks=range(len(a_values)),yticklabels=a_values,xlabel='Length multiplier',title=f'α={alpha:g}')
axes[0].set_ylabel('Outer A width fraction')
fig.colorbar(im,ax=axes,shrink=.85,label='Failing state references (out of 11)')
display(fig);plt.close(fig)
print('Counterfactual exclusions only; nothing is deleted:')
wd_exclusions=pd.read_csv(WD_OUT/'exclusion_scenarios.csv')
display(wd_exclusions[wd_exclusions.alpha.isin([.71,.6])].sort_values(['alpha','policy'],ascending=[False,True]))

### Does excluding an extreme combination change the conclusion?

At **α=0.71**, every failure is confined to **outer A=0.05 and multiplier=3**, repeated at all four network extents. The nested parent, crossover parent and nested main-island-collapse child fail; simple/full remains individually feasible. This yields **12 failed unique networks and 24 failed parent–child comparisons**. Removing the four complete configurations would remove 28 comparison slots, including four feasible simple/full comparisons, and leave 136/136 configurations feasible.

At **α=0.6**, all 140 extension configurations, 603 unique networks and 980 pair clocks pass. All 2,187 configurations of the original factorial grid pass at every tested exponent. At α=0.8, failures include A=0.05 at multipliers 2 and 3; constant kb also fails less-lengthened narrow branches, so not every exponent's failures are restricted to the single most extreme combination.

A restricted main domain plus separately reported stress tests is an option. Exclusion would need a scientific scope statement; a numerical failure alone does not show the geometry is unrealistic. Keep the stress-test records, and do not silently remove them or use their omission to imply universal robustness.

### What do the real base networks add?

These are stored geometry/reference checks for four selected examples, not depth observations or a representative survey of rivers. The width axis below is **branch width / BASED reference width**, not the fraction at a bifurcation. The shaded box marks the approximate parent-geometry range of the synthetic embedded extension (width ratio 0.025–1; slope ratio 1/3–1). It is a geometric comparison, not a hydraulic validity region.

At α=0.6, 03 and 56 pass; 21 and 55 fail. The controlling slow links in 21 and 55 are 228m and 175m wide, with slopes approximately 3.1×10⁻⁶ and 6.5×10⁻⁶. These are about 1.2% and 3.2% of their reference slopes, much lower than the long synthetic branches' one-third reference slope. Their source is `raw_wse`; inspect that provenance and the head-drop estimates before treating these slopes as either valid extremes or errors. Width-refresh markers remain for 03/21/56.

The real widths support retaining substantial branch-width contrasts in sensitivity tests. The stronger real slope heterogeneity means that resolving the synthetic extremes cannot certify real-network feasibility. This evidence helps define the test domain and identify data checks; it does not calibrate α.

In [ ]:
wd_real_edges=pd.read_csv(WD_OUT/'real_edge_preflight.csv',dtype={'example':str})
wd_real_controllers=pd.read_csv(WD_OUT/'real_controllers.csv',dtype={'example':str})
real_domain=pd.read_csv(WD_OUT/'real_geometry_domain.csv',dtype={'example':str})
display(wd_real.pivot(index='example',columns='alpha',values='feasible').reindex(columns=WD_ALPHA))
display(real_domain.round(5))
controller_view=wd_real_controllers[(wd_real_controllers.alpha==.6)&(wd_real_controllers.role=='slow')].copy()
controller_view['implied_link_head_drop_m']=controller_view.slope_used*controller_view.link_length_m
display(controller_view[['example','id_link','wid_adj_routing','depth_m','slope_used','slope_over_reference','implied_link_head_drop_m','slope_section_ref','slope_source_method','dt_lower_seconds','dt_upper_seconds','network_feasible']])
from matplotlib.patches import Rectangle
fig,axes=plt.subplots(2,2,figsize=(12,8),constrained_layout=True)
for ax,ex in zip(axes.flat,['03','21','55','56']):
    d=wd_real_edges[(wd_real_edges.example==ex)&(wd_real_edges.alpha==.6)]
    ax.add_patch(Rectangle((.025,1/3),.975,2/3,facecolor='#dcfce7',edgecolor='#16a34a',alpha=.6,label='Synthetic parent range'))
    ax.scatter(d.width_over_reference,d.slope_over_reference,s=16,color='#64748b',alpha=.55,label='Real links')
    for flag,color,label in [('controls_lower','#dc2626','Slow controller'),('controls_upper','#2563eb','Fast controller')]:
        z=d[d[flag]];ax.scatter(z.width_over_reference,z.slope_over_reference,s=80,facecolors='none',edgecolors=color,linewidths=2,label=label)
    ax.set(xscale='log',yscale='log',xlim=(.02,2),ylim=(.008,5),xlabel='Branch width / reference width',ylabel='Branch slope / reference slope',title=f'Example {ex} · α=0.6');ax.grid(alpha=.15)
handles,labels=axes[0,0].get_legend_handles_labels();fig.legend(handles,labels,loc='outside lower center',ncol=4,fontsize=9)
display(fig);plt.close(fig)

### Updated decision

α=0.6 is a useful intermediate candidate: it resolves all tested synthetic extension preflights and the example-56 base conflict while moderating depths less than 0.5. α=0.71 remains a literature-motivated candidate with a clearly identified extreme-combination boundary. Neither is selected as a default.

For baseline simple/full under the common clock, α=0.6 gives lag shift −22.91%, event-duration change −1.10%, and peak-excess change +1.17%. In the critical long-narrow 12.5m branch it gives depth 0.379m, but Manning-implied reference flow is only 0.409m³/s versus the prescribed steady allocation of 25m³/s. The depth/allocation consistency issue therefore remains.

The expanded focused experiment completed **273 primary comparisons and 273 common-clock controls, 568 unique routes**, with all conservation and plateau checks passing. Old-method hydrograph/comparison parity still passes. Next assess the low-slope real controllers and independent depth evidence before selecting an exponent or excluding configurations. No production defaults, real hydrographs, or experiment membership changed.

**Subsequent decision, 30 September:** α=0.65 is now selected for synthetics. Real-case selection remains deferred pending the new slope-based runs. The preceding candidate discussion records the earlier diagnostic stage.

## Fine alpha threshold check · 30 September 2026

Reference **HYD-2026-09-30-ALPHA-THRESHOLD**. User requested the first failure between 0.6 and 0.71. A 0.001-spaced scan plus targeted values covers all 2,187 original-grid and 140 extension configurations; bisection resolves the four limiting configurations. No hydrographs were rerun or exponent adopted.

With the existing integer-timestep scheduler, the first boundary is approximately **alpha=0.669901503**. Alpha=0.6699 passes; 0.66991 fails. On a 0.01 grid, 0.66 passes and 0.67 is the first failure. The limiting combination remains A=.05, multiplier=3, at all four extents. The original grid passes throughout the scanned range.

| Alpha | Limiting continuous dt interval (s) | Integer choices | Outcome |
|---|---|---:|---|
| .60 | 277.879–330.152 | 53 | pass |
| .65 | 314.238–330.152 | 16 | pass |
| .66 | 322.061–330.152 | 8 | pass |
| 2/3 | 327.385–330.152 | 3 | pass |
| .6699 | 329.999–330.152 | 1 | pass |
| .67 | 330.080–330.152 | 0 | fail |
| .71 | 364.201–330.152 | 0 | fail |

At .67 the continuous interval is nonempty but contains no integer second, explaining failure under the actual scheduler. Choosing the last passing exponent would tune to this numerical boundary. Alpha=.65 or .66 retains more margin; 2/3 is feasible here but with only 328/329/330 s available. None of these checks supplies independent physical justification for an exponent. Conditions remain n=.035, X=.1, 200m target/floor, fixed BASED references and existing celerity caps.

Reproduce `investigations/width_depth_alpha_threshold.py`; scan, summary and bisection brackets are in `outputs/width_depth_scaling/alpha_threshold/`.


## Selected synthetic setting · α=0.65 · 30 September 2026

**HYD-2026-09-30-SELECTED-ALPHA065.** The user selected α=0.65 after the literature-motivated sensitivity and numerical boundary check. Keep the existing BASED reference width/depth, n=0.035, X=0.1, 200m segmentation/floor, and width allocation; apply local kb consistently to celerity and the Chezy resistance calculation. The choice retains numerical margin in the tested synthetic domain; it is not claimed to be an independently calibrated physical optimum.

All 2,187 original-grid and 140 extension configurations pass the selected preflight. The most restrictive synthetic configuration admits integer timesteps 315–330 seconds. The table below shows the actual selected focused routing results. The subsequently completed full factorial and extension rerun is reported in the next section; the current sensitivity and predictor notebooks now use that selected version.

**Real cases:** the decision explicitly waits for new runs based on updated slopes. The older stored-base failure checks remain historical diagnostics; no real-case exponent or default is selected here. The branch-depth/allocated-discharge mismatch remains an acknowledged limitation for the synthetic hydraulic interpretation.

In [ ]:
selected_alpha=wd_manifest['selected_synthetic_alpha']
assert selected_alpha==.65
print('Selected synthetic alpha:',selected_alpha,'| reference:',wd_manifest['selection_reference'])
print('Real cases:',wd_manifest['real_case_decision'])
display(wd[wd.alpha==selected_alpha].query("clock_mode=='common_clock'")[['case','child_state','alpha','dt_seconds','resistance_predictor']+WD_Y].round(4))
display(wd_grid[wd_grid.alpha==selected_alpha].groupby('dataset').feasible.agg(passing='sum',tested='count'))

## Full selected synthetic rerun · 30 September 2026

The versioned full grid and all 140 extension configurations are rerun at alpha=.65 with BASED reference kb and n=.035. Geometry/forcing/allocation are held as planned. The summary below refers to those actual routes. Existing real-case audits remain historical; the real decision awaits updated slopes.

In [ ]:
import sys
sys.path.insert(0,str(ROOT/'synthetic_experiments/investigations'))
from analysis_paths import results_root
ACTIVE_RUN=results_root()/'full_grid_island'
ACTIVE_EXTENSION=results_root()/'embedded_width_length_extension'
assert json.loads((ACTIVE_RUN/'manifest.json').read_text())['config']['selected_hydraulic_reference']['depth_alpha']==.65
selected_version=ACTIVE_RUN.parent
print('Selected results:',selected_version)
display(pd.read_csv(selected_version/'quality_summary.csv'))
display(pd.read_csv(selected_version/'old_new_response_summary.csv').round(3))

### Hydraulic reference coordinates under the selected full-grid setup

These diagnostics are recalculated with each job's BASED reference and local α=.65 depth. Allocated reference Q is the width-propagated inlet base/peak flow, **not the actual local transient peak**. Manning velocity is computed from the same depth, n=.035 and slope before celerity clipping. Comparing these definitions exposes the remaining hydraulic-closure mismatch.

The green area is only the repository's provisional F²≤0.9 / T⁺≥80 screen, using the 12h rise time as a proxy. It is not the complete published wave-regime diagram or a proof of RAPID applicability. Repeated branch/run rows are not independent river observations. No old real-pilot points are mixed into this selected-run figure; real-case interpretation awaits the new slope-based runs.

In [ ]:
SELECTED_HYD=ACTIVE_RUN.parent/'hydraulic_applicability'
selected_hyd=pd.read_parquet(SELECTED_HYD/'reaches.parquet')
display(pd.read_csv(SELECTED_HYD/'summary.csv'))
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for ax,mode,title in zip(axes,['allocated','manning'],['Velocity from allocated peak Q / area','Manning velocity at assigned depth']):
    q=selected_hyd[selected_hyd.reference_flow=='peak']
    ax.scatter(q[mode+'_tplus'],q[mode+'_f2'],s=3,alpha=.12,color='#2563eb',rasterized=True)
    ax.set(xscale='log',yscale='log',xlabel='T⁺ (12h rise-time proxy)',ylabel='F²',title=title)
    ax.axvline(80,color='#16a34a',ls='--');ax.axhline(.9,color='#16a34a',ls='--')
    xmin,xmax=ax.get_xlim();ymin,ymax=ax.get_ylim()
    if xmax>80 and ymin<.9:ax.fill_between([80,xmax],ymin,.9,color='#dcfce7',alpha=.5,zorder=0)
    ax.grid(alpha=.15)
fig.savefig(SELECTED_HYD/'selected_reference_coordinates.png',dpi=160)
display(fig);plt.close(fig)